In [ ]:
# Colab 2026.07：先在「變更執行階段類型」選 NVIDIA GPU。
import re
import subprocess
import sys
toolkit = subprocess.check_output(["nvcc", "--version"], text=True)
major = re.search(r"release (\d+)\.", toolkit).group(1)
if major not in {"12", "13"}:
    raise RuntimeError("此範例針對 CUDA 12 或 13")
subprocess.run([sys.executable, "-m", "pip", "install", "pyscf",
                f"gpu4pyscf-cuda{major}x", f"cutensor-cu{major}"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv"], check=True)

## 重繪書圖
`medium_old/github/pyscf-xtb-benchmark-main/pyscf/results/*_pyscf_results.json`。下方內嵌四份檔案的 `scf_time_s` 和五態區塊 `tddft.tddft_time_s`。硬體型號未隨結果儲存。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# 支援在專案根目錄、book 子目錄或獨立 Colab 工作階段執行。
here = Path.cwd()
book_root = next((p for p in (here, *here.parents) if (p / "outline.yml").is_file()), None)
if book_root is None and (here / "book" / "outline.yml").is_file():
    book_root = here / "book"
fig_dir = (book_root or here) / "figures"
fig_dir.mkdir(exist_ok=True)
seconds = {'p_nitroaniline': (25.4, 189.7), 'naphthalene': (28.7, 270.6), 'caffeine': (52.7, 522.9), 'PrK2': (195.9, 2130.1)}

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.0))
y = np.arange(len(seconds))
scf, td = np.array(list(seconds.values())).T
for values, offset, color, label in [(scf, -0.18, "#176d78", "SCF"),
                                      (td, 0.18, "#ab6627", "TDDFT (5 states)")]:
    ax.barh(y + offset, values - 10, left=10, height=0.32, color=color, label=label)
    for pos, value in zip(y + offset, values):
        ax.text(value * 1.06, pos, f"{value:g}", va="center", fontsize=9)
ax.set(xscale="log", xlim=(10, 4500), xlabel="Elapsed time (s, log scale)",
       yticks=y, yticklabels=["p-Nitroaniline", "Naphthalene", "Caffeine", "PrK2"])
ax.invert_yaxis()
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(fig_dir / "05-07-gpu-accelerated-dft.png", dpi=160, bbox_inches="tight")
plt.show()

## 新計算（Colab 2026.07）
執行後會產生本次的獨立結果。原資料及書圖保留在上方。

In [ ]:
from importlib.metadata import version
from time import perf_counter
import cupy as cp
from pyscf import dft, gto, lib

for package in ["pyscf", f"gpu4pyscf-cuda{major}x", f"cutensor-cu{major}"]:
    print(package, version(package))
lib.num_threads(2)
print("CPU threads", lib.num_threads())
# medium_old/github/pyscf-xtb-benchmark-main/molecules/caffeine_opt.xyz（去掉 XYZ 標頭）
atoms = """C           -3.25335095715195       -1.13440034953138        0.02297844623363
N           -2.28856966751145       -0.07289290919864        0.01151439031831
C           -2.50377852586451        1.25383893666376        0.00514648861900
N           -1.39808970613959        1.96820236818213        0.00161891906270
C           -0.40316020027698        1.05414277697402        0.00656589207477
C           -0.92200268383242       -0.23254712810968        0.01295662947717
C           -0.09564522183725       -1.40102859593253        0.01565557019529
O           -0.46716037783360       -2.55917635338782        0.01731399523978
N            1.25824410522029       -1.05193834358584        0.01542579592990
C            1.81093007970281        0.22290765024438        0.01172872738321
O            3.01189590498833        0.38199876692355        0.01330325383396
N            0.93545442515880        1.28901313588031        0.00602038659959
C            1.43838102601104        2.64512283834645        0.00156025272617
C            2.22830797438830       -2.13512191024276        0.01955688609036
H           -4.25570708498658       -0.71202423441823       -0.01446272729005
H           -3.13856720240500       -1.73332738248100        0.92780139692249
H           -3.09470708851278       -1.78982476060900       -0.83438416283469
H           -3.48422934772898        1.68686670482021        0.00242692362401
H            2.52507216296540        2.59651583499982        0.02324102093262
H            1.06359289895089        3.17747369373985        0.87526308646663
H            1.09986989054926        3.16011049847260       -0.89722777761662
H            1.67515736196778       -3.07141390905147        0.01888489422954
H            2.85749486859399       -2.06412269334979        0.90632348203995
H            2.86293736558420       -2.06586463534894       -0.86341177025772"""
mol = gto.M(atom=atoms, basis="6-31+g(d,p)", unit="Angstrom", charge=0, spin=0)


In [ ]:
def calculate(use_gpu):
    mf = dft.RKS(mol)
    mf.xc = "m06-2x"
    mf.grids.atom_grid = (99, 590)
    mf.grids.prune = None
    mf.conv_tol = 1e-9
    if use_gpu:
        mf = mf.to_gpu()
    if use_gpu:
        cp.cuda.runtime.deviceSynchronize()
    start = perf_counter()
    energy = float(mf.kernel())
    if use_gpu:
        cp.cuda.runtime.deviceSynchronize()
    elapsed = perf_counter() - start
    if not mf.converged:
        raise RuntimeError("SCF 未收斂")
    occ = cp.asnumpy(mf.mo_occ) if use_gpu else mf.mo_occ
    energies = cp.asnumpy(mf.mo_energy) if use_gpu else mf.mo_energy
    gap_ev = (energies[occ == 0].min() - energies[occ > 0].max()) * 27.2114
    return energy, gap_ev, elapsed

cpu = calculate(False)
gpu = calculate(True)
print("device | total energy (Eh) | orbital gap (eV) | elapsed (s)")
for name, (energy, gap, elapsed) in [("CPU", cpu), ("GPU", gpu)]:
    print(f"{name:3s} {energy:18.9f} {gap:15.5f} {elapsed:12.2f}")
print("Energy difference (Eh)", gpu[0] - cpu[0])
print("CPU time / GPU time", cpu[2] / gpu[2])